# 05 — The departure backtest (Phase 5)

**What?** The trial of the whole system. For every player a Big-5 club sold in the last ten
summers, we rewind to that summer, let the system build a replacement shortlist using only
what was knowable then, and check how those players actually did afterwards — against the
player the club really bought, against a simple goals+assists rule, and against "buy the most
expensive one you can afford".

**Why?** This is the project's success test: does the machine buy better than reality?

**How to read the results.** Every shortlist is judged on three things nobody can argue with:
minutes actually played per million euros, real goals+assists per 90, and how the player's
market value changed — always per euro spent. A shortlist **wins a case** when it beats its
opponent on at least two of the three. "Case-win 0.59" means it won 59% of head-to-heads; the
brackets are an 80% uncertainty range from resampling the cases. **Frozen** means computed
only from seasons before the sale — the code raises an error if anything peeks past the
freeze. One thing evolves as the notebook goes: *how to sort a shortlist* is itself an open
question here, tested head-to-head in Steps 4–9 until a final answer stands.


## Contents

- Step 1 — The backtest population: which departures qualify?
  - Step 1 — what we got
- Step 2 — The bar, the pool, and affordability at freeze time
  - Step 2 — what we got
- Step 3 — P(≥ bar) from the intervals
  - Step 3 — what we got
- Step 4 — The orderings: how do you sort a shortlist?
  - Step 4 — what we got
- Step 5 — Grading against reality
  - Step 5 — what we got
- Step 6 — Can defender quality be measured better? Percentages and possession-adjustment
  - Step 6 — what we got
- Step 7 — The similarity × output blend
  - Step 7 — what we got
- Step 8 — The Moneyball formula: expected production, and production per euro
  - Step 8 — what we got: the Moneyball formula wins
- Step 9 — The formula across all six positions
  - Step 9 — what we got: one formula for the outfield, keepers exempt
- Step 10 — The package reproduces the backtest


## Step 1 — The backtest population: which departures qualify?

**What?** All paid departures from Big-5 clubs, summer windows 2015 → 2024: how many have a
proper statistical season at the selling club the year before (without one, there is nothing
to replace *from*), split by season and position, and under different definitions of
"a smaller club". Also: how often the selling club made an identifiable same-position signing
that summer — the strongest possible opponent for the system.

**Why?** The population defines what the backtest can claim. Every choice here (which sales
count, whether to filter by club size) is made from the count tables below, not assumed.


In [1]:
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 300)

from scout import config
from scout.data import clubelo, reep, understat
from scout.data import transfermarkt as tm_loader
from scout.identity import build_team_lineage, load_overrides
from scout.panel import elo as elo_panel
from scout.panel import identity, market, stints

LEAGUE_TO_COMP = {league: comp for comp, league in config.BIG5.items()}
COMPS = list(config.BIG5) + list(config.FEEDERS)

contrib = pd.DataFrame(json.load(open(config.MODELS / "phase2_contribution.json")))
tm_panel = tm_loader.load_player_club_seasons(COMPS, list(config.SEASONS))
tm_clubs = tm_panel[["club_id", "club_name", "competition_id"]].drop_duplicates()
us = understat.load("player_season")
us["competition_id"] = us.league.map(LEAGUE_TO_COMP)
lineage = build_team_lineage(
    tm_clubs,
    {
        "understat": us[["competition_id", "team"]]
        .drop_duplicates()
        .rename(columns={"team": "team_name"})
    },
    load_overrides("teams"),
)
us_ids = (
    identity.resolve_provider(
        "understat", us, identity.transfermarkt_side(tm_panel), lineage, reep.load_people()
    )
    .drop_duplicates("provider_id")
    .set_index("provider_id")
    .tm_player_id
)
contrib["tm_player_id"] = contrib.player_id.astype(int).astype(str).map(us_ids)

st = stints.build(COMPS, list(config.SEASONS))
st["tm_player_id"] = st.tm_player_id.astype(str)
season_value = st.sort_values(["minutes", "club_id"], ascending=[False, True]).drop_duplicates(
    ["tm_player_id", "season"]
)[["tm_player_id", "season", "club_id", "competition_id", "value_july", "value_age_days_july"]]
print(len(contrib), "contribution rows |", len(st), "stints")

21478 contribution rows | 77794 stints


In [2]:
# paid departures from Big-5 clubs (classified moves, so paid loans are excluded)
con = tm_loader.connect()
moves = market.add_cost(market.classify(con))
moves["transfer_season"] = 2000 + moves.transfer_season.str.slice(0, 2).astype(int)
moves["month"] = pd.to_datetime(moves.transfer_date).dt.month

big5_clubs = con.execute(
    "SELECT CAST(club_id AS INTEGER) AS club_id, domestic_competition_id AS competition_id "
    "FROM clubs WHERE domestic_competition_id IN ('GB1','ES1','IT1','L1','FR1')"
).df()

sales = moves[
    (moves.kind == "paid")
    & moves.from_club_id.isin(big5_clubs.club_id)
    & moves.transfer_season.between(2015, 2024)
].copy()
sales["summer"] = ~sales.month.isin([1, 2, 3])  # winter sales close the current season
sales["prev_season"] = sales.transfer_season - sales.summer.astype(int)
sales["tm_player_id"] = sales.player_id.astype(str)
sales = sales.merge(
    big5_clubs.rename(columns={"club_id": "from_club_id", "competition_id": "sell_comp"}),
    on="from_club_id",
)
print(len(sales), "paid departures | window split:", sales.summer.value_counts().to_dict())
print("by month:", sales.month.value_counts().sort_index().to_dict())

panel_row = tm_panel[["tm_player_id", "club_id", "season", "minutes"]].rename(
    columns={"club_id": "from_club_id", "season": "prev_season"}
)
panel_row["tm_player_id"] = panel_row.tm_player_id.astype(str)
sales = sales.merge(panel_row, on=["tm_player_id", "from_club_id", "prev_season"], how="left")

dep_role = (
    contrib.dropna(subset=["tm_player_id"])
    .sort_values(["minutes", "competition_id", "player_id"], ascending=[False, True, True])
    .drop_duplicates(["tm_player_id", "season"])[["tm_player_id", "season", "role"]]
    .rename(columns={"season": "prev_season", "role": "dep_role"})
)
sales = sales.merge(dep_role, on=["tm_player_id", "prev_season"], how="left")

summer = sales[sales.summer]
print(
    f"summer sales {len(summer)} | panel row at the club the season before: "
    f"{summer.minutes.notna().mean():.1%} | with a >=600-min role: {summer.dep_role.notna().mean():.1%}"
)
cases = summer.dropna(subset=["dep_role"]).copy()
print("\ncases with a role, by season:")
print(cases.groupby("transfer_season").size().to_string())
print("\nby role:", cases.dep_role.value_counts().to_dict())

2781 paid departures | window split: {True: 2328, False: 453}
by month: {1: 365, 2: 80, 3: 8, 4: 3, 5: 1, 6: 2, 7: 1305, 8: 806, 9: 185, 10: 23, 11: 3}
summer sales 2328 | panel row at the club the season before: 61.5% | with a >=600-min role: 53.0%

cases with a role, by season:
transfer_season
2015     85
2016     82
2017    118
2018    100
2019    133
2020     85
2021     99
2022    164
2023    200
2024    168

by role: {'CM': 303, 'W': 288, 'CB': 241, 'ST': 212, 'FB': 190}


In [3]:
# the selling club's tier at the sale summer: Elo on 1 July + squad-value rank
squad_value = (
    season_value.groupby(["competition_id", "season", "club_id"])
    .value_july.sum()
    .rename("squad_value")
    .reset_index()
)
squad_value = squad_value[squad_value.competition_id.isin(config.BIG5)].copy()
squad_value["squad_rank"] = squad_value.groupby(["competition_id", "season"]).squad_value.rank(
    ascending=False
)

elo_names = elo_panel.club_elo_names(list(config.BIG5), list(config.SEASONS))
club_seasons = squad_value[["competition_id", "season", "club_id"]]
elo_rows = []
for club_id, name in elo_names.items():
    hist = clubelo.fetch_club(name)
    if hist.empty:
        continue
    mine = club_seasons[club_seasons.club_id == club_id]
    if mine.empty:
        continue
    dates = pd.Series([f"{s}-07-01" for s in mine.season])
    for (_, row), e in zip(mine.iterrows(), elo_panel.elo_on_dates(hist, dates)):
        elo_rows.append((row.competition_id, row.season, club_id, e))
club_elo = pd.DataFrame(elo_rows, columns=["competition_id", "season", "club_id", "elo_july"])
club_elo["elo_rank"] = club_elo.groupby(["competition_id", "season"]).elo_july.rank(ascending=False)

tier = squad_value.merge(club_elo, on=["competition_id", "season", "club_id"], how="outer")
cases_t = cases.merge(
    tier.rename(columns={"club_id": "from_club_id", "season": "transfer_season"}),
    on=["from_club_id", "transfer_season"],
    how="left",
)
print(
    "tier coverage on cases:",
    {c: f"{cases_t[c].notna().mean():.1%}" for c in ["elo_rank", "squad_rank"]},
)

candidates = {
    "no filter": cases_t.elo_rank.notna(),
    "elo_rank > 4": cases_t.elo_rank > 4,
    "elo_rank > 6": cases_t.elo_rank > 6,
    "squad_rank > 4": cases_t.squad_rank > 4,
    "squad_rank > 6": cases_t.squad_rank > 6,
}
print("\ncases kept per filter candidate, by role:")
kept = pd.DataFrame(
    {name: cases_t[mask].dep_role.value_counts() for name, mask in candidates.items()}
)
print(kept.to_string())
print("\ncases per season x role, elo_rank > 6 (the tightest):")
tight = cases_t[candidates["elo_rank > 6"]]
print(tight.pivot_table(index="transfer_season", columns="dep_role", aggfunc="size").to_string())
print(
    "\nseason x role cells under 10 cases:",
    f"elo>6 {(tight.pivot_table(index='transfer_season', columns='dep_role', aggfunc='size') < 10).to_numpy().sum()}",
    f"| no filter {(cases_t.pivot_table(index='transfer_season', columns='dep_role', aggfunc='size') < 10).to_numpy().sum()}",
    "of",
    10 * 6,
    "cells",
)

tier coverage on cases: {'elo_rank': '81.7%', 'squad_rank': '81.7%'}

cases kept per filter candidate, by role:
          no filter  elo_rank > 4  elo_rank > 6  squad_rank > 4  squad_rank > 6
dep_role                                                                       
CM              254           175           143             176             141
W               234           157           129             160             122
CB              191           144           115             142             108
ST              170           128           103             131             104
FB              159           101            87             102              83

cases per season x role, elo_rank > 6 (the tightest):
dep_role         CB  CM  FB  ST   W
transfer_season                    
2015              2  11   8   8   9
2016              5   9   7   4  12
2017              7  13   5  15  13
2018              7  12  13   3  10
2019             12  14  11   8  12
2020              9  

In [4]:
# how many cases have an identifiable actual replacement (same role bought the same summer)?
incoming = moves[
    moves.kind.isin(["paid", "free", "undisclosed"])
    & moves.to_club_id.isin(big5_clubs.club_id)
    & moves.transfer_season.between(2015, 2024)
    & ~moves.month.isin([1, 2, 3])
].copy()
incoming["tm_player_id"] = incoming.player_id.astype(str)
in_role = incoming.merge(
    dep_role.rename(columns={"dep_role": "in_role"}),
    left_on=["tm_player_id", "transfer_season"],
    right_on=["tm_player_id", "prev_season"],
    how="left",
    suffixes=("", "_r"),
)
in_role["prev2"] = in_role.transfer_season - 1
in_role = in_role.merge(
    dep_role.rename(columns={"dep_role": "in_role_prev", "prev_season": "prev2"}),
    on=["tm_player_id", "prev2"],
    how="left",
)
in_role["role_known"] = in_role.in_role_prev.fillna(in_role.in_role)
arrivals = in_role.dropna(subset=["role_known"])[
    ["to_club_id", "transfer_season", "role_known", "kind", "cost"]
]

case_keys = cases_t[["from_club_id", "transfer_season", "dep_role", "tm_player_id"]]
matched = case_keys.merge(
    arrivals.rename(columns={"to_club_id": "from_club_id", "role_known": "dep_role"}),
    on=["from_club_id", "transfer_season", "dep_role"],
    how="left",
)
per_case = matched.groupby(["from_club_id", "transfer_season", "tm_player_id"]).agg(
    n_in=("kind", "count"), n_paid=("cost", lambda c: c.notna().sum())
)
print(
    f"cases with any same-role arrival that summer: {(per_case.n_in > 0).mean():.1%} | "
    f"with a costed same-role arrival: {(per_case.n_paid > 0).mean():.1%} (n={len(per_case)})"
)

cases with any same-role arrival that summer: 44.7% | with a costed same-role arrival: 44.7% (n=1234)


In [5]:
# 1b — keepers (contrib is outfield-only), the missing tiers, and the elo>4 middle option
keepers = pd.DataFrame(json.load(open(config.MODELS / "phase2_keepers.json")))
keepers["tm_player_id"] = keepers.player_id.astype(int).astype(str).map(us_ids)
gk_role = (
    keepers.dropna(subset=["tm_player_id"])
    .sort_values(["minutes", "competition_id", "player_id"], ascending=[False, True, True])
    .drop_duplicates(["tm_player_id", "season"])
    .assign(dep_role="GK")
    .rename(columns={"season": "prev_season"})[["tm_player_id", "prev_season", "dep_role"]]
)
gk_cases = (
    summer[summer.dep_role.isna()]
    .drop(columns=["dep_role"])
    .merge(gk_role, on=["tm_player_id", "prev_season"])
)
print(
    f"GK cases recovered: {len(gk_cases)} | by season:",
    gk_cases.groupby("transfer_season").size().to_dict(),
)

cases_all = pd.concat([cases, gk_cases], ignore_index=True)
cases_all_t = cases_all.merge(
    tier.rename(columns={"club_id": "from_club_id", "season": "transfer_season"}),
    on=["from_club_id", "transfer_season"],
    how="left",
)

# who has no tier? sellers not in a Big-5 top flight at the sale summer
panel_clubs = tm_panel[["club_id", "season", "competition_id"]].drop_duplicates()
sellers = cases_all_t.merge(
    panel_clubs.rename(
        columns={
            "club_id": "from_club_id",
            "season": "transfer_season",
            "competition_id": "comp_at_sale",
        }
    ),
    on=["from_club_id", "transfer_season"],
    how="left",
)
no_tier = sellers[sellers.elo_rank.isna()]
print(
    f"\ncases without a tier: {len(no_tier)} of {len(sellers)} | "
    f"seller in NO panel league that season (relegated out): "
    f"{no_tier.comp_at_sale.isna().mean():.1%} | still in a Big-5 league: "
    f"{no_tier.comp_at_sale.isin(list(config.BIG5)).mean():.1%}"
)

for thresh in (4, 6):
    sub = cases_all_t[cases_all_t.elo_rank > thresh]
    tab = sub.pivot_table(index="transfer_season", columns="dep_role", aggfunc="size").fillna(0)
    print(
        f"\nelo_rank > {thresh}: {len(sub)} cases | season x role cells under 10: "
        f"{(tab < 10).to_numpy().sum()} of {tab.size}"
    )
    if thresh == 4:
        print(tab.astype(int).to_string())
tiered = cases_all_t[cases_all_t.elo_rank.notna()]
tab0 = tiered.pivot_table(index="transfer_season", columns="dep_role", aggfunc="size").fillna(0)
print(
    f"\nno filter: {len(tiered)} tiered cases | cells under 10: "
    f"{(tab0 < 10).to_numpy().sum()} of {tab0.size} (GK thin everywhere by nature)"
)

GK cases recovered: 83 | by season: {2015: 5, 2016: 8, 2017: 8, 2018: 12, 2019: 11, 2020: 8, 2021: 8, 2022: 7, 2023: 10, 2024: 6}

cases without a tier: 248 of 1317 | seller in NO panel league that season (relegated out): 100.0% | still in a Big-5 league: 0.0%

elo_rank > 4: 756 cases | season x role cells under 10: 24 of 60
dep_role         CB  CM  FB  GK  ST   W
transfer_season                        
2015              5  16   8   3   9  13
2016              8   9   8   3   7  15
2017             10  14   5   3  16  18
2018              8  15  15   7   5  12
2019             12  18  11   6  14  17
2020             10  12   5   7   7  13
2021             17  15   6   6   9  13
2022             25  26  15   5  21  10
2023             23  32  14   7  21  27
2024             26  18  14   4  19  19

elo_rank > 6: 618 cases | season x role cells under 10: 28 of 60

no filter: 1069 tiered cases | cells under 10: 12 of 60 (GK thin everywhere by nature)


### Step 1 — what we got

**The population: 1,069 replayable departures, 2015–2024 summers.** Of 2,781 paid departures
from Big-5 clubs, 84% are summer-window sales (winter is 16%, almost all January — not replayed:
a mid-season emergency buy is a different product). 61.5% of summer sales have a panel row at
the selling club the season before, 53% a ≥600-minute role season (the 43%/47% gap is the known
population caveat: youth, reserves, loanees). Goalkeepers had dropped out of the first count —
the role table is outfield-only — and were recovered from the keeper artifact: +83 GK cases.
248 cases (18.8%) belong to sellers relegated out of the top flight by the sale summer —
**excluded**: a second-division club is not running a Big-5 replacement search; reported, not
hidden.

**The "smaller club" filter: none — the tier is a reported split instead.** Hard filters starve
the per-season tables: elo_rank > 4 leaves 24 of 60 season × role cells under 10 cases,
elo_rank > 6 leaves 28; the unfiltered population leaves 12 (all driven by GK, thin by nature —
GK verdicts are pooled across seasons, never per season). Squad-value rank keeps counts nearly
identical to Elo rank (756 vs 750 at >4) and adds nothing. So: every tiered seller is graded,
and the headline additionally quotes the **elo_rank > 6 subset (618 cases) as the
smaller-club split**, per role, pooled over seasons where thin. Rejected: elo>4 / elo>6 /
squad-rank as population definitions.

**The strongest baseline exists for ~45% of cases:** a costed same-role arrival at the selling
club the same summer — the club's actual answer, with its price. The budget rule is settled in
Step 2 on this subset; the departing player's sale price is the fallback constraint for the
rest.


## Step 2 — The bar, the pool, and affordability at freeze time

**What?** For every case: the candidate pool as of the sale summer — players with a
≥600-minute season in the same position the year before, priced (market value at 1 July),
filtered to those *shaped like the departing player* (the similarity gate), and affordable.
Candidates for the budget rule and for how tight the similarity gate should be, decided on
pool sizes and worked examples.

**Why?** The pool defines what a shortlist can contain. The similarity gate is where the
football knowledge lives — it keeps a destroyer-midfielder search full of destroyer
midfielders. Profiles are rebuilt per summer from strictly earlier seasons, so the gate never
sees the future.


In [6]:
# Step 2a — frozen profiles + eligibility per sale summer (strictly past seasons only)
from scout.models import fit as fit_model
from scout.models import similarity as similarity_model
from scout.panel import player_match

pm = player_match.build()
pm["competition_id"] = pm.league.map(LEAGUE_TO_COMP)
shots = understat.load("shots")
shots["competition_id"] = shots.league.map(LEAGUE_TO_COMP)
players_tm = tm_loader.load_table("players")[["player_id", "name"]]
players_tm["tm_player_id"] = players_tm.player_id.astype(str)

gk_universe = keepers.dropna(subset=["tm_player_id"]).assign(role="GK")
cols = [
    "competition_id",
    "season",
    "player_id",
    "tm_player_id",
    "role",
    "minutes",
    "point",
    "lo",
    "hi",
]
universe = pd.concat(
    [contrib.dropna(subset=["tm_player_id"])[cols], gk_universe[cols]], ignore_index=True
)
price = season_value[["tm_player_id", "season", "value_july"]]

frozen = {}
for s in sorted(cases_all_t.transfer_season.unique()):
    past = pm[pm.season < s]
    prof = similarity_model.profile(past, shots[shots.season < s])
    elig = fit_model.eligible_slots(fit_model.role_shares(past[past.season == s - 1]))
    frozen[s] = {"profiles": prof[prof.season == s - 1], "eligible": elig}
    print(s, "| profiles at s-1:", len(frozen[s]["profiles"]), "| eligible rows:", len(elig))

2015 | profiles at s-1: 1955 | eligible rows: 2371
2016 | profiles at s-1: 1927 | eligible rows: 2454


2017 | profiles at s-1: 1935 | eligible rows: 2432


2018 | profiles at s-1: 1926 | eligible rows: 2403


2019 | profiles at s-1: 1904 | eligible rows: 2367


2020 | profiles at s-1: 1866 | eligible rows: 2406


2021 | profiles at s-1: 1946 | eligible rows: 2439


2022 | profiles at s-1: 1939 | eligible rows: 2479


2023 | profiles at s-1: 1929 | eligible rows: 2431


2024 | profiles at s-1: 1900 | eligible rows: 2386


In [7]:
# Step 2b — per-case pools under the similarity gates and affordability rules
role_all = pd.concat([dep_role, gk_role], ignore_index=True)
in_all = incoming.merge(
    role_all.rename(columns={"dep_role": "in_role"}),
    left_on=["tm_player_id", "transfer_season"],
    right_on=["tm_player_id", "prev_season"],
    how="left",
)
in_all["prev2"] = in_all.transfer_season - 1
in_all = in_all.merge(
    role_all.rename(columns={"dep_role": "in_role_prev", "prev_season": "prev2"}),
    on=["tm_player_id", "prev2"],
    how="left",
)
in_all["role_known"] = in_all.in_role_prev.fillna(in_all.in_role)
arr_cost = (
    cases_all_t[["from_club_id", "transfer_season", "dep_role", "tm_player_id"]]
    .merge(
        in_all.dropna(subset=["role_known", "cost"])[
            ["to_club_id", "transfer_season", "role_known", "cost"]
        ].rename(columns={"to_club_id": "from_club_id", "role_known": "dep_role"}),
        on=["from_club_id", "transfer_season", "dep_role"],
        how="left",
    )
    .groupby(["from_club_id", "transfer_season", "tm_player_id"])
    .cost.max()
    .rename("arrival_cost")
)

records = []
for s, grp in cases_all_t.groupby("transfer_season"):
    prof = frozen[s]["profiles"]
    elig = frozen[s]["eligible"]
    price_s = (
        price[price.season == s]
        .drop_duplicates("tm_player_id")
        .set_index("tm_player_id")
        .value_july
    )
    uni_s = universe[universe.season == s - 1]
    for case in grp.itertuples():
        me = uni_s[(uni_s.tm_player_id == case.tm_player_id) & (uni_s.role == case.dep_role)]
        rec = {
            "transfer_season": s,
            "tm_player_id": case.tm_player_id,
            "dep_role": case.dep_role,
            "from_club_id": case.from_club_id,
            "sale_fee": case.cost,
            "has_profile": False,
        }
        if not me.empty:
            my_us_id = me.sort_values(
                ["minutes", "competition_id"], ascending=[False, True]
            ).player_id.iloc[0]
            ok_ids = set(elig[elig[f"can_{case.dep_role}"]].player_id)
            cand = uni_s[uni_s.player_id.isin(ok_ids) & (uni_s.tm_player_id != case.tm_player_id)]
            cand = (
                cand.sort_values(
                    ["minutes", "competition_id", "player_id"], ascending=[False, True, True]
                )
                .drop_duplicates("tm_player_id")
                .copy()
            )
            cand["value"] = cand.tm_player_id.map(price_s)
            cand = cand.dropna(subset=["value"])
            sub = (
                prof[prof.role == case.dep_role].drop_duplicates("player_id").set_index("player_id")
            )
            if my_us_id in sub.index:
                rec["has_profile"] = True
                F = sub[similarity_model.FEATURES].to_numpy(float)
                x = sub.loc[my_us_id, similarity_model.FEATURES].to_numpy(float)
                dist = pd.Series(np.sqrt(((F - x) ** 2).sum(axis=1)), index=sub.index)
                cand["dist"] = cand.player_id.map(dist)
                cand = cand.dropna(subset=["dist"])
                rec["n_pool"] = len(cand)
                for gate_name, gated in [
                    ("top25", cand.nsmallest(25, "dist")),
                    ("top50", cand.nsmallest(50, "dist")),
                    ("pct25", cand[cand.dist <= cand.dist.quantile(0.25)]),
                    ("pct50", cand[cand.dist <= cand.dist.quantile(0.50)]),
                ]:
                    rec[f"n_{gate_name}"] = len(gated)
                    rec[f"n_{gate_name}_sale"] = (gated.value <= case.cost).sum()
                ac = arr_cost.get((case.from_club_id, s, case.tm_player_id), np.nan)
                rec["arrival_cost"] = ac
                top50 = cand.nsmallest(50, "dist")
                rec["n_top50_arrival"] = (top50.value <= ac).sum() if pd.notna(ac) else np.nan
        records.append(rec)
pools = pd.DataFrame(records)
print(
    f"cases: {len(pools)} | departing player has a frozen profile row: {pools.has_profile.mean():.1%}"
)
print("\nmedian pool sizes by role (eligible+priced, then gated; _sale = also under the sale fee):")
cols = ["n_pool", "n_top25", "n_top50", "n_pct25", "n_pct50", "n_top50_sale", "n_top50_arrival"]
print(pools.groupby("dep_role")[cols].median().round(0).to_string())
print("\nshare of cases with >=20 candidates:")
print(
    pools.groupby("dep_role")[["n_top25", "n_top50", "n_pct25", "n_pct50", "n_top50_sale"]]
    .apply(lambda g: (g >= 20).mean().round(2))
    .to_string()
)

cases: 1317 | departing player has a frozen profile row: 100.0%

median pool sizes by role (eligible+priced, then gated; _sale = also under the sale fee):
          n_pool  n_top25  n_top50  n_pct25  n_pct50  n_top50_sale  n_top50_arrival
dep_role                                                                           
CB         320.0     25.0     50.0     80.0    160.0          36.0             29.0
CM         357.0     25.0     50.0     90.0    179.0          30.0             28.0
FB         291.0     25.0     50.0     73.0    146.0          36.0             28.0
GK         114.0     25.0     50.0     29.0     57.0          30.0             25.0
ST         201.0     25.0     50.0     51.0    101.0          34.0             27.0
W          334.0     25.0     50.0     84.0    167.0          29.0             29.0

share of cases with >=20 candidates:
          n_top25  n_top50  n_pct25  n_pct50  n_top50_sale
dep_role                                                  
CB            1.0

In [8]:
# Step 2c — worked examples: the biggest smaller-club sales, three roles
examples = (
    cases_all_t[(cases_all_t.elo_rank > 6) & cases_all_t.dep_role.isin(["W", "CM", "CB"])]
    .sort_values(["cost", "tm_player_id"], ascending=[False, True])
    .drop_duplicates("dep_role")
)
name_of = players_tm.drop_duplicates("tm_player_id").set_index("tm_player_id")["name"]
for case in examples.itertuples():
    s = case.transfer_season
    prof, elig = frozen[s]["profiles"], frozen[s]["eligible"]
    price_s = (
        price[price.season == s]
        .drop_duplicates("tm_player_id")
        .set_index("tm_player_id")
        .value_july
    )
    uni_s = universe[universe.season == s - 1]
    me = uni_s[(uni_s.tm_player_id == case.tm_player_id) & (uni_s.role == case.dep_role)]
    my_us_id = me.sort_values(
        ["minutes", "competition_id"], ascending=[False, True]
    ).player_id.iloc[0]
    ok_ids = set(elig[elig[f"can_{case.dep_role}"]].player_id)
    cand = uni_s[uni_s.player_id.isin(ok_ids) & (uni_s.tm_player_id != case.tm_player_id)]
    cand = (
        cand.sort_values(["minutes", "competition_id", "player_id"], ascending=[False, True, True])
        .drop_duplicates("tm_player_id")
        .copy()
    )
    cand["value"] = cand.tm_player_id.map(price_s)
    sub = prof[prof.role == case.dep_role].drop_duplicates("player_id").set_index("player_id")
    F = sub[similarity_model.FEATURES].to_numpy(float)
    x = sub.loc[my_us_id, similarity_model.FEATURES].to_numpy(float)
    dist = pd.Series(np.sqrt(((F - x) ** 2).sum(axis=1)), index=sub.index)
    cand["dist"] = cand.player_id.map(dist)
    shortlist = (
        cand.dropna(subset=["dist", "value"])[lambda d: d.value <= case.cost]
        .nsmallest(10, "dist")
        .assign(name=lambda d: d.tm_player_id.map(name_of))
    )
    print(
        f"=== {name_of.get(case.tm_player_id, case.tm_player_id)} ({case.dep_role}) sold summer {s} "
        f"for {case.cost / 1e6:.0f} M EUR (seller elo rank {case.elo_rank:.0f}) ==="
    )
    print(
        shortlist[["name", "competition_id", "dist", "value", "point"]]
        .assign(value=lambda d: (d.value / 1e6).round(1))
        .round(2)
        .to_string(index=False)
    )
    print()

=== Jack Grealish (W) sold summer 2021 for 118 M EUR (seller elo rank 11) ===
             name competition_id  dist  value  point
   Vincenzo Grifo             L1  2.23   12.0   0.53
   Kingsley Coman             L1  2.46   65.0   0.47
Aleksandr Golovin            FR1  2.56   28.0   0.59
    Heung-min Son            GB1  2.70   85.0   0.53
   Filip Djuricic            IT1  2.72    9.0   0.54
     Jadon Sancho             L1  2.75  100.0   0.59
     Ross Barkley            GB1  2.77   20.0   0.45
    Daniel Didavi             L1  2.81    1.8   0.50
   Nicola Sansone            IT1  2.93    3.8   0.48
         Raphinha            GB1  3.03   30.0   0.56

=== Declan Rice (CM) sold summer 2023 for 117 M EUR (seller elo rank 11) ===
            name competition_id  dist  value  point
Pedro Chirivella            FR1  1.23    6.0   0.09
   Mateo Kovacic            GB1  1.31   38.0   0.17
       Ivan Ilić            IT1  1.47   19.0   0.13
       Dion Lopy            FR1  1.57    4.0   0.13
 

### Step 2 — what we got

**Every departing player has a frozen profile row (100.0%),** so no case is lost to the gate.
Median eligible + priced pools per role: CB 320, CM 357, FB 291, W 334, ST 201, GK 114.

**Gate: top-50 most similar within the role.** The percentile gates scale with the role's size
(pct50 admits a median 57–179 candidates — half the role is not a "profile-plausible set"),
while top-50 is constant, keeps ≥20 candidates in 100% of cases in every role, and still leaves
a median 29–36 candidates after the budget cap. Rejected: top-25 (too thin once the budget
bites), pct25/pct50 (pool-size dependent, loose for big roles).

**Budget rule: cost ≤ the departing player's sale fee, universally; the actual-arrival budget
as the head-to-head subset.** "Replace him with the money you got for him" applies to 100% of
cases and keeps ≥20 affordable candidates in 69–81% of cases per role; the stricter "what the
club actually paid its same-role signing" exists for ~45% of cases (median 25–29 affordable
candidates there) and becomes the strongest-baseline comparison in Step 5, not the population
definition. Uncapped is kept as a sensitivity only.

**The examples behave like scouting, not like a goals leaderboard.** Grealish 2021 → Grifo,
Coman, Golovin, Sancho, Raphinha (wide creators at €4–100m against his €118m fee). Rice 2023 →
Kovacic, Locatelli, Koné, Ricci — Rice-shaped deep midfielders with npxG+xA points of
0.07–0.19: the pool is shaped by profile, exactly as the fit-first design intends. Maguire
2019 → ball-playing CBs including Diego Carlos (€13m) and Upamecano (€30m) against his €87m
fee — two names the market later repriced dramatically upward.


## Step 3 — P(≥ bar) from the intervals

**What?** For output-ranked positions the shortlist sorts by the probability that a candidate
delivers at least the departing player's level ("the bar"). Three ways to turn a candidate's
uncertainty interval into that probability, fitted on seasons ≤ 2019 and judged on 2020+.

**Why?** A probability is only useful if it is honest: when the method says 70%, it should
come true about 70% of the time. The reliability tables below decide the method; ties go to
the simplest.


In [9]:
# Step 3a — candidate-vs-bar pairs from the real case pools (W, ST, GK; Big-5 candidates)
out_all = pd.concat([contrib.dropna(subset=["tm_player_id"]), gk_universe], ignore_index=True)
out_all = out_all.sort_values(
    ["minutes", "competition_id", "player_id"], ascending=[False, True, True]
).drop_duplicates(["tm_player_id", "season", "role"])
nxt_out = (
    out_all.assign(season=out_all.season - 1).set_index(["tm_player_id", "season", "role"]).per90
)

pair_frames = []
for s, grp in cases_all_t[cases_all_t.dep_role.isin(["W", "ST", "GK"])].groupby("transfer_season"):
    prof, elig = frozen[s]["profiles"], frozen[s]["eligible"]
    price_s = (
        price[price.season == s]
        .drop_duplicates("tm_player_id")
        .set_index("tm_player_id")
        .value_july
    )
    uni_s = universe[universe.season == s - 1]
    for case in grp.itertuples():
        me = uni_s[(uni_s.tm_player_id == case.tm_player_id) & (uni_s.role == case.dep_role)]
        if me.empty:
            continue
        me = me.sort_values(["minutes", "competition_id"], ascending=[False, True])
        bar, my_us_id = me.point.iloc[0], me.player_id.iloc[0]
        ok_ids = set(elig[elig[f"can_{case.dep_role}"]].player_id)
        cand = uni_s[
            uni_s.player_id.isin(ok_ids)
            & (uni_s.tm_player_id != case.tm_player_id)
            & uni_s.competition_id.isin(list(config.BIG5))
        ]
        cand = (
            cand.sort_values(
                ["minutes", "competition_id", "player_id"], ascending=[False, True, True]
            )
            .drop_duplicates("tm_player_id")
            .copy()
        )
        cand["value"] = cand.tm_player_id.map(price_s)
        cand = cand.dropna(subset=["value"])
        sub = prof[prof.role == case.dep_role].drop_duplicates("player_id").set_index("player_id")
        if my_us_id not in sub.index:
            continue
        F = sub[similarity_model.FEATURES].to_numpy(float)
        x = sub.loc[my_us_id, similarity_model.FEATURES].to_numpy(float)
        dist = pd.Series(np.sqrt(((F - x) ** 2).sum(axis=1)), index=sub.index)
        cand["dist"] = cand.player_id.map(dist)
        gated = cand.dropna(subset=["dist"])
        gated = gated[gated.value <= case.cost].nsmallest(50, "dist").copy()
        gated["bar"] = bar
        gated["transfer_season"] = s
        gated["dep_role"] = case.dep_role
        lookup = pd.MultiIndex.from_frame(gated[["tm_player_id", "season", "role"]])
        gated["out_next"] = nxt_out.reindex(lookup).to_numpy()
        pair_frames.append(
            gated[
                [
                    "transfer_season",
                    "dep_role",
                    "tm_player_id",
                    "point",
                    "lo",
                    "hi",
                    "bar",
                    "out_next",
                ]
            ]
        )
pairs = pd.concat(pair_frames, ignore_index=True)
print(
    len(pairs),
    "candidate-bar pairs |",
    f"realised next season available: {pairs.out_next.notna().mean():.1%}",
)
pairs = pairs.dropna(subset=["out_next"]).copy()
pairs["sd"] = (pairs.hi - pairs.lo) / (2 * 1.2816)
print("kept:", len(pairs), "| by role:", pairs.dep_role.value_counts().to_dict())

26103 candidate-bar pairs | realised next season available: 64.8%
kept: 16908 | by role: {'W': 8156, 'ST': 6500, 'GK': 2252}


In [10]:
# Step 3b — three probability methods; tails fitted <= 2019, reliability judged on 2020+
from scipy import stats

gen = out_all.copy()
gen["sd"] = (gen.hi - gen.lo) / (2 * 1.2816)
lookup = pd.MultiIndex.from_frame(gen[["tm_player_id", "season", "role"]])
gen["out_next"] = nxt_out.reindex(lookup).to_numpy()
train = gen[gen.season <= 2018].dropna(subset=["out_next"])
z_train = ((train.out_next - train.point) / train.sd).to_numpy()
t_df, t_loc, t_scale = stats.t.fit(z_train)
print(
    f"projection errors <=2019: n={len(z_train)} | sd(z)={z_train.std():.2f} | "
    f"fitted t: df={t_df:.1f}, loc={t_loc:.3f}, scale={t_scale:.2f}"
)

t_val = ((pairs.bar - pairs.point) / pairs.sd).to_numpy()
z_sorted = np.sort(z_train)
pairs["p_norm"] = 1 - stats.norm.cdf(t_val)
pairs["p_t"] = 1 - stats.t.cdf(t_val, t_df, loc=t_loc, scale=t_scale)
pairs["p_emp"] = 1 - np.searchsorted(z_sorted, t_val, side="left") / len(z_sorted)
pairs["hit"] = pairs.out_next >= pairs.bar

bins = np.arange(0, 1.01, 0.1)
for label, frame in [
    ("out of sample (2020+)", pairs[pairs.transfer_season >= 2020]),
    ("in sample (<2020)", pairs[pairs.transfer_season < 2020]),
]:
    print(f"\n=== reliability, {label} (n={len(frame)}) ===")
    scores = {}
    for m in ["p_norm", "p_t", "p_emp"]:
        cut = pd.cut(frame[m], bins)
        tab = frame.groupby(cut, observed=True).agg(
            predicted=(m, "mean"), realised=("hit", "mean"), n=("hit", "size")
        )
        gap = (tab.predicted - tab.realised).abs()
        scores[m] = float((gap * tab.n).sum() / tab.n.sum())
        if m == "p_norm":
            print(tab.round(2).to_string())
    print("weighted |predicted - realised|:", {k: round(v, 3) for k, v in scores.items()})

projection errors <=2019: n=5895 | sd(z)=7.28 | fitted t: df=2.4, loc=-0.005, scale=0.78

=== reliability, out of sample (2020+) (n=8754) ===
            predicted  realised     n
p_norm                               
(0.0, 0.1]       0.04      0.14   987
(0.1, 0.2]       0.15      0.19   855
(0.2, 0.3]       0.25      0.25   914
(0.3, 0.4]       0.35      0.31  1117
(0.4, 0.5]       0.45      0.39  1192
(0.5, 0.6]       0.55      0.51  1209
(0.6, 0.7]       0.65      0.61  1025
(0.7, 0.8]       0.75      0.68   746
(0.8, 0.9]       0.85      0.78   504
(0.9, 1.0]       0.94      0.94   204
weighted |predicted - realised|: {'p_norm': 0.046, 'p_t': 0.048, 'p_emp': 0.048}

=== reliability, in sample (<2020) (n=8154) ===
            predicted  realised     n
p_norm                               
(0.0, 0.1]       0.04      0.10  1064
(0.1, 0.2]       0.15      0.21   890
(0.2, 0.3]       0.25      0.22   978
(0.3, 0.4]       0.35      0.29  1059
(0.4, 0.5]       0.45      0.35  1153
(0.5, 

### Step 3 — what we got

**The normal approximation wins by the tie-break.** On 16,908 candidate-vs-bar pairs from the
real case pools (W 8,156, ST 6,500, GK 2,252; Big-5 candidates), reliability out of sample
(2020+, n=8,754): weighted |predicted − realised| = 0.046 for the normal on the Phase 2
interval, 0.048 for the fitted Student-t (df 2.4 — the heavy tails are real but don't help
here), 0.048 for the raw empirical error distribution. Effectively a three-way tie → the
simplest method, `P = 1 − Φ((bar − point)/sd)` with sd read off the 80% interval, is the
decided form. Rejected: fitted-t and empirical (equal accuracy, extra machinery).

**Honest wrinkles, reported:** the method is mildly overconfident in the upper-middle range
(says 75% → happens 68%; says 85% → 78%) and pessimistic at the very bottom (says 4% → 14%) —
a known shrinkage signature; the weighted error of ~0.05 is quoted wherever probabilities are
shown. Calibration can only use candidates with a ≥600-minute following season (64.8% of
pairs) — the probability is therefore "P(≥ bar), given he plays"; the playing part is the
availability model's job, kept separate by design.


## Step 4 — The orderings: how do you sort a shortlist?

**What?** Every case's gated, affordable pool gets everything an ordering could use — age,
expected minutes next season (a model refitted per summer on strictly past data), the
probability of clearing the bar, defensive-activity levels, last season's goals+assists, the
price. Then the candidate sorting rules are defined: similarity-led variants (most like the
departed player, alone or combined with price or expected minutes) and output-led variants
(probability of clearing the bar, alone or per euro). Each family's sub-variant is chosen on
pre-2020 cases only and then frozen, so the later grading years stay untouched by the tuning.

**Why?** "Which number should order a defender's shortlist" was the most contested question
of the project. This notebook settles it the only honest way: every rule becomes a tournament
entrant and history grades them all.


In [11]:
# Step 4a — enriched candidate pools per case (all roles), frozen per summer
players_full = tm_loader.load_table("players")[["player_id", "date_of_birth"]]
players_full["tm_player_id"] = players_full.player_id.astype(str)
birth_year = (
    players_full.drop_duplicates("tm_player_id")
    .set_index("tm_player_id")
    .date_of_birth.pipe(pd.to_datetime)
    .dt.year
)

# availability history (player_id level, target = next season TM minutes, as in train_phase3)
minutes_rows = (
    pd.concat([contrib.dropna(subset=["tm_player_id"]), gk_universe], ignore_index=True)
    .sort_values(["minutes", "competition_id", "player_id"], ascending=[False, True, True])
    .drop_duplicates(["player_id", "season"])
    .copy()
)
minutes_rows["age"] = minutes_rows.season + 1 - minutes_rows.tm_player_id.map(birth_year)
from scout.models import availability as avail_model
from scout.models import trajectory as traj_model

hist = avail_model.history(minutes_rows[["player_id", "season", "minutes", "age", "role"]])
tm_minutes = st.groupby(["tm_player_id", "season"]).minutes.sum().rename("target").reset_index()
tm_minutes["season"] = tm_minutes.season - 1
pid_to_tm = minutes_rows.drop_duplicates("player_id").set_index("player_id").tm_player_id
hist["tm_player_id"] = hist.player_id.map(pid_to_tm)
hist = hist.merge(tm_minutes, on=["tm_player_id", "season"], how="left")

# trajectory pairs (for the frozen role curve per summer)
one_role = (
    pd.concat([contrib.dropna(subset=["tm_player_id"]), gk_universe], ignore_index=True)
    .sort_values(["minutes", "competition_id", "player_id"], ascending=[False, True, True])
    .drop_duplicates(["player_id", "role", "season"])
    .copy()
)
one_role["age"] = one_role.season + 1 - one_role.tm_player_id.map(birth_year)
nxt_pt = one_role.assign(season=one_role.season - 1)[["player_id", "role", "season", "point"]]
traj_pairs = one_role.merge(
    nxt_pt.rename(columns={"point": "point_next"}), on=["player_id", "role", "season"]
)

# defensive activity per 90 (Sofascore canonical), z within role-season
from scout.data import sofascore
from scout.panel import workrate

ss = sofascore.load()
ss_lineage = build_team_lineage(
    tm_clubs,
    {"sofascore": ss[["competition_id", "team_name"]].drop_duplicates()},
    load_overrides("teams"),
)
ss_ids = (
    identity.resolve_provider(
        "sofascore", ss, identity.transfermarkt_side(tm_panel), ss_lineage, reep.load_people()
    )
    .drop_duplicates("provider_id")
    .set_index("provider_id")
    .tm_player_id
)
wr = pd.concat(
    [
        ss[["competition_id", "season", "sofascore_player_id", "minutesPlayed"]],
        workrate.sofascore_per90(ss),
    ],
    axis=1,
)
wr["tm_player_id"] = wr.sofascore_player_id.astype(int).astype(str).map(ss_ids)
DEF_ACTIONS = [
    "tackles",
    "interceptions",
    "clearances",
]  # ballRecovery only exists from 2023-24 (Phase 1 Step 6)
wr_def = (
    wr.dropna(subset=["tm_player_id"])
    .assign(minutes=lambda d: pd.to_numeric(d.minutesPlayed))
    .sort_values(
        ["minutes", "competition_id", "sofascore_player_id"], ascending=[False, True, True]
    )
    .drop_duplicates(["tm_player_id", "season"])[["tm_player_id", "season"] + DEF_ACTIONS]
)

# naive G+A per 90 (Understat season totals)
ga = us.copy()
ga["tm_player_id"] = ga.player_id.astype(int).astype(str).map(us_ids)
ga = (
    ga.dropna(subset=["tm_player_id"])
    .groupby(["tm_player_id", "season"])[["goals", "assists", "minutes"]]
    .sum()
    .reset_index()
)
ga["ga90"] = (ga.goals + ga.assists) / ga.minutes * 90

pool_frames = []
for s, grp in cases_all_t.groupby("transfer_season"):
    prof, elig = frozen[s]["profiles"], frozen[s]["eligible"]
    price_s = (
        price[price.season == s]
        .drop_duplicates("tm_player_id")
        .set_index("tm_player_id")
        .value_july
    )
    uni_s = universe[universe.season == s - 1]
    curve_s = traj_model.role_curve(traj_pairs[traj_pairs.season <= s - 2])
    train_rows = hist[hist.season <= s - 2].fillna({"target": 0.0})
    hist_s = hist[hist.season == s - 1].copy()
    if len(train_rows) >= 500:
        avail_fit_s = avail_model.fit(train_rows)
        hist_s["expected_minutes"] = avail_model.predict(avail_fit_s, hist_s)
    else:  # summer 2015: no season has a known target yet -> the lag1 baseline
        hist_s["expected_minutes"] = hist_s.lag1
    exp_min = hist_s.drop_duplicates("player_id").set_index("player_id").expected_minutes
    wr_s = wr_def[wr_def.season == s - 1].set_index("tm_player_id")[DEF_ACTIONS]
    ga_s = ga[ga.season == s - 1].drop_duplicates("tm_player_id").set_index("tm_player_id").ga90
    for case in grp.itertuples():
        me = uni_s[(uni_s.tm_player_id == case.tm_player_id) & (uni_s.role == case.dep_role)]
        if me.empty:
            continue
        me = me.sort_values(["minutes", "competition_id"], ascending=[False, True])
        bar, my_us_id = me.point.iloc[0], me.player_id.iloc[0]
        ok_ids = set(elig[elig[f"can_{case.dep_role}"]].player_id)
        cand = uni_s[uni_s.player_id.isin(ok_ids) & (uni_s.tm_player_id != case.tm_player_id)]
        cand = (
            cand.sort_values(
                ["minutes", "competition_id", "player_id"], ascending=[False, True, True]
            )
            .drop_duplicates("tm_player_id")
            .copy()
        )
        cand["value"] = cand.tm_player_id.map(price_s)
        cand = cand.dropna(subset=["value"])
        sub = prof[prof.role == case.dep_role].drop_duplicates("player_id").set_index("player_id")
        if my_us_id not in sub.index:
            continue
        F = sub[similarity_model.FEATURES].to_numpy(float)
        x = sub.loc[my_us_id, similarity_model.FEATURES].to_numpy(float)
        dist = pd.Series(np.sqrt(((F - x) ** 2).sum(axis=1)), index=sub.index)
        cand["dist"] = cand.player_id.map(dist)
        gated = cand.dropna(subset=["dist"])
        gated = gated[gated.value <= case.cost].nsmallest(50, "dist").copy()
        if gated.empty:
            continue
        gated["age"] = s - gated.tm_player_id.map(birth_year)
        gated["sd"] = (gated.hi - gated.lo) / (2 * 1.2816)
        from scipy import stats as _st

        gated["p_bar"] = 1 - _st.norm.cdf((bar - gated.point) / gated.sd)
        gated["expected_minutes"] = gated.player_id.map(exp_min)
        for a in DEF_ACTIONS:
            gated[a] = gated.tm_player_id.map(wr_s[a])
        gated["ga90"] = gated.tm_player_id.map(ga_s)
        gated["case_id"] = f"{case.from_club_id}_{s}_{case.tm_player_id}"
        gated["transfer_season"] = s
        gated["dep_role"] = case.dep_role
        gated["sale_fee"] = case.cost
        gated["bar"] = bar
        pool_frames.append(gated)
cand_pools = pd.concat(pool_frames, ignore_index=True)
cand_pools["def_sum"] = cand_pools[DEF_ACTIONS].sum(axis=1, min_count=3)
cand_pools["def_z"] = cand_pools.groupby(["dep_role", "transfer_season"]).def_sum.transform(
    lambda x: (x - x.mean()) / (x.std() if x.std() > 0 else 1.0)
)
print(
    len(cand_pools),
    "pool rows across",
    cand_pools.case_id.nunique(),
    "cases | enrichment coverage:",
    {
        c: f"{cand_pools[c].notna().mean():.1%}"
        for c in ["age", "p_bar", "expected_minutes", "def_z", "ga90"]
    },
)
(config.ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
# realised outcomes (post-freeze by design: used only for grading, never in an ordering)
mins_map = tm_minutes.set_index(["tm_player_id", "season"]).target
cand_pools["out_minutes"] = mins_map.reindex(
    pd.MultiIndex.from_arrays([cand_pools.tm_player_id, cand_pools.transfer_season - 1])
).to_numpy()
ga_next = ga[ga.minutes >= 600].set_index(["tm_player_id", "season"]).ga90
cand_pools["out_ga90"] = ga_next.reindex(
    pd.MultiIndex.from_arrays([cand_pools.tm_player_id, cand_pools.transfer_season])
).to_numpy()
val_next = (
    price.drop_duplicates(["tm_player_id", "season"])
    .set_index(["tm_player_id", "season"])
    .value_july
)
cand_pools["out_value_next"] = val_next.reindex(
    pd.MultiIndex.from_arrays([cand_pools.tm_player_id, cand_pools.transfer_season + 1])
).to_numpy()
print(
    "outcome coverage:",
    {
        c: f"{cand_pools[c].notna().mean():.1%}"
        for c in ["out_minutes", "out_ga90", "out_value_next"]
    },
)
cand_pools.to_parquet(config.ROOT / "data" / "processed" / "phase5_cand_pools.parquet")
print("saved data/processed/phase5_cand_pools.parquet (repo root)")

/Users/mihailandreev/football-player-scouting/src/scout/models/availability.py:36: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  return smf.ols(FORMULA, data=train).fit()


/Users/mihailandreev/football-player-scouting/src/scout/models/availability.py:36: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  return smf.ols(FORMULA, data=train).fit()


60411 pool rows across 1304 cases | enrichment coverage: {'age': '100.0%', 'p_bar': '100.0%', 'expected_minutes': '100.0%', 'def_z': '93.0%', 'ga90': '100.0%'}
outcome coverage: {'out_minutes': '100.0%', 'out_ga90': '76.7%', 'out_value_next': '76.7%'}
saved data/processed/phase5_cand_pools.parquet (repo root)


In [12]:
# Step 4b — the tournament entrants; sub-variants tuned on pre-2020 cases only
FIT_ROLES, OUT_ROLES = ["CB", "FB", "CM"], ["W", "ST", "GK"]


def ordered(pool, ordering, n=5):
    p = pool
    if ordering == "f1":
        key = p.dist.rank()
    elif ordering == "f2":
        key = (p.dist.rank() + p.value.rank()) / 2
    elif ordering == "f3":
        key = (p.dist.rank() + (-p.expected_minutes).rank()) / 2
    elif ordering == "o1":
        key = (-(p.p_bar / (p.value / 1e6))).rank()
    elif ordering == "o2":
        key = (-p.p_bar).rank()
    elif ordering == "defence":
        key = (-p.def_z).rank()
    elif ordering == "market":
        key = (-p.value).rank()
    elif ordering == "naive":
        key = (-p.ga90).rank()
    return p.assign(_k=key).dropna(subset=["_k"]).nsmallest(n, "_k")


def scores(pool_subset, ordering, n=5):
    rows = []
    for cid, g in pool_subset.groupby("case_id"):
        top = ordered(g, ordering, n)
        if len(top) < n:
            continue
        rows.append(
            {
                "case_id": cid,
                "minutes_per_meur": top.out_minutes.fillna(0).sum() / (top.value.sum() / 1e6),
                "ga90_mean": top.out_ga90.mean(),
                "value_ratio": top.out_value_next.sum() / top.value.sum(),
            }
        )
    return pd.DataFrame(rows)


pre = cand_pools[cand_pools.transfer_season <= 2019]
print("=== fit-first variants (CB/FB/CM), pre-2020 cases, top-5 per case ===")
fit_tab = {}
for v in ["f1", "f2", "f3"]:
    sc = scores(pre[pre.dep_role.isin(FIT_ROLES)], v)
    fit_tab[v] = sc[["minutes_per_meur", "ga90_mean", "value_ratio"]].mean()
fit_tab = pd.DataFrame(fit_tab).T
print(fit_tab.round(3).to_string())
winner_fit = (fit_tab.rank(ascending=False).mean(axis=1)).idxmin()
print("winner by mean rank across the three outcome columns:", winner_fit)

print("\n=== output variants (W/ST/GK), pre-2020 cases, top-5 per case ===")
out_tab = {}
for v in ["o1", "o2"]:
    sc = scores(pre[pre.dep_role.isin(OUT_ROLES)], v)
    out_tab[v] = sc[["minutes_per_meur", "ga90_mean", "value_ratio"]].mean()
out_tab = pd.DataFrame(out_tab).T
print(out_tab.round(3).to_string())
winner_out = (out_tab.rank(ascending=False).mean(axis=1)).idxmin()
print("winner by mean rank across the three outcome columns:", winner_out)

print("\nworked shortlists under the chosen orderings (top-5):")
for case in examples.itertuples():
    cid = f"{case.from_club_id}_{case.transfer_season}_{case.tm_player_id}"
    g = cand_pools[cand_pools.case_id == cid]
    if g.empty:
        continue
    ordering = winner_fit if case.dep_role in FIT_ROLES else winner_out
    top = ordered(g, ordering, 5).assign(name=lambda d: d.tm_player_id.map(name_of))
    print(
        f"\n--- {name_of.get(case.tm_player_id)} ({case.dep_role}, {case.transfer_season}, "
        f"fee {case.cost / 1e6:.0f} M) — ordering {ordering} ---"
    )
    print(
        top[["name", "dist", "value", "p_bar", "expected_minutes", "out_minutes", "out_ga90"]]
        .assign(value=lambda d: (d.value / 1e6).round(1))
        .round(2)
        .to_string(index=False)
    )

=== fit-first variants (CB/FB/CM), pre-2020 cases, top-5 per case ===


    minutes_per_meur  ga90_mean  value_ratio
f1           584.630      0.117        1.015
f2          1268.871      0.105        0.970
f3           570.477      0.121        1.116
winner by mean rank across the three outcome columns: f3

=== output variants (W/ST/GK), pre-2020 cases, top-5 per case ===


    minutes_per_meur  ga90_mean  value_ratio
o1          1193.151      0.337        1.026
o2           467.537      0.408        1.083
winner by mean rank across the three outcome columns: o2

worked shortlists under the chosen orderings (top-5):

--- Jack Grealish (W, 2021, fee 118 M) — ordering o2 ---
              name  dist  value  p_bar  expected_minutes  out_minutes  out_ga90
        Sadio Mané  4.01   85.0   0.71           2178.16       2825.0      0.57
Christopher Nkunku  3.32   43.0   0.65           1238.71       2733.0      1.08
   Raheem Sterling  3.91   90.0   0.65           1933.38       2127.0      0.76
   Mikel Oyarzabal  3.39   70.0   0.61           2179.65       1676.0      0.64
      Jadon Sancho  2.75  100.0   0.57           1980.39       1901.0      0.29

--- Declan Rice (CM, 2023, fee 117 M) — ordering f3 ---
            name  dist  value  p_bar  expected_minutes  out_minutes  out_ga90
Manuel Locatelli  1.68   30.0   0.54           2272.90       3011.0      0.15
Pe

### Step 4 — what we got

**Fit-first ordering (CB/FB/CM) = f3: mean rank of similarity-to-X and expected minutes.** On
pre-2020 cases only (top-5 per case, three outcome columns): f3 wins realised output (0.121
G+A/90) and value growth (×1.116) and takes the mean-rank verdict; f1 (pure similarity) is
worse on two of three; f2 (similarity + cheapness) triples minutes-per-euro (1,269 vs 570) by
buying cheap squad players but has the worst output and value growth — the money discipline
belongs in the budget cap, not the sort. Rejected: f1, f2.

**Output ordering (W/ST/GK) = o2: P(≥ bar) alone, within the budget.** o2 beats o1 (P per
euro) on realised output (0.408 vs 0.337 G+A/90) and value growth (1.083 vs 1.026); o1 wins
only minutes-per-euro, the same cheap-player artefact. The affordability cap already enforces
"per euro"; inside it, quality sorts best. Rejected: o1.

**The tuning rule itself, stated:** variants compared on the three model-independent outcome
columns with equal weight (mean rank); pre-2020 cases only, frozen before any 2020+ case is
scored. Comparison entrants (defence-activity, market, naive, actual signing) are definitions,
not tuned. The worked shortlists behave like scouting: Grealish 2021 → Mané, Nkunku (€43m,
realised 1.08 G+A/90), Sterling, Oyarzabal, Sancho; Rice 2023 → Locatelli, Chirivella, Koné,
Ilić, Jensen (2,100–3,011 realised minutes); Maguire 2019 → Diego Carlos (€13m, 3,135
minutes), Keane, Izzo, Pezzella, Ginter.


## Step 5 — Grading against reality

**What?** Every case's frozen top-5 shortlist, scored on the three outcome columns and
compared case by case against the club's actual same-position signing (at its actual fee),
the market ordering ("most expensive affordable"), and the naive goals+assists rule — plus,
for the defensive positions, a head-to-head between the similarity-led and output-led
sorting rules and a pure defending-stats rule. 2020+ is the out-of-tuning window; the
smaller-club subset is reported alongside.

**Why?** This is the success metric of the whole project — and the first data verdict on the
sorting question.


In [13]:
# Step 5a — the club's actual answer, and every entrant's per-case scores
FIT_ORD, OUT_ORD = "f3", "o2"
COLS = ["minutes_per_meur", "ga90_mean", "value_ratio"]

case_meta = cand_pools.drop_duplicates("case_id")[
    ["case_id", "transfer_season", "dep_role", "sale_fee"]
].copy()
# from_club_id lives only inside the case_id string ("club_season_player")
case_meta["from_club_id"] = case_meta.case_id.str.split("_").str[0].astype("int64")
case_meta = case_meta.set_index("case_id")

arr = in_all.dropna(subset=["role_known", "cost"])[
    ["to_club_id", "transfer_season", "role_known", "tm_player_id", "cost"]
].rename(columns={"to_club_id": "from_club_id", "role_known": "dep_role"})
arr["from_club_id"] = arr.from_club_id.astype("int64")
actual = (
    case_meta.reset_index()
    .merge(arr, on=["from_club_id", "transfer_season", "dep_role"])
    .sort_values(["cost", "tm_player_id"], ascending=[False, True])
    .drop_duplicates("case_id")
)
actual["out_minutes"] = mins_map.reindex(
    pd.MultiIndex.from_arrays([actual.tm_player_id, actual.transfer_season - 1])
).to_numpy()
actual["out_ga90"] = ga_next.reindex(
    pd.MultiIndex.from_arrays([actual.tm_player_id, actual.transfer_season])
).to_numpy()
actual["out_value_next"] = val_next.reindex(
    pd.MultiIndex.from_arrays([actual.tm_player_id, actual.transfer_season + 1])
).to_numpy()
actual_sc = pd.DataFrame(
    {
        "case_id": actual.case_id,
        "minutes_per_meur": actual.out_minutes.fillna(0) / (actual.cost / 1e6),
        "ga90_mean": actual.out_ga90,
        "value_ratio": actual.out_value_next / actual.cost,
    }
).set_index("case_id")
print(f"cases with an actual same-role costed signing: {len(actual_sc)} of {case_meta.shape[0]}")

FIT_SET = {"CB", "FB", "CM"}
entrants = {"actual": actual_sc}
for name, ordering_of, roles in [
    ("model", lambda r: FIT_ORD if r in FIT_SET else OUT_ORD, None),
    ("output_all", lambda r: "o2", None),
    ("market", lambda r: "market", None),
    ("naive", lambda r: "naive", None),
    ("defence", lambda r: "defence", {"CB", "FB"}),
]:
    frames = []
    for role, g in cand_pools.groupby("dep_role"):
        if roles and role not in roles:
            continue
        frames.append(scores(g, ordering_of(role)))
    entrants[name] = pd.concat(frames).set_index("case_id")
print({k: len(v) for k, v in entrants.items()})

cases with an actual same-role costed signing: 577 of 1304


{'actual': 577, 'model': 1284, 'output_all': 1284, 'market': 1284, 'naive': 1284, 'defence': 401}


In [14]:
# Step 5b — verdict tables: paired case wins with bootstrap 80% intervals
rng = np.random.default_rng(0)


def case_wins(a, b, ids):
    both = a.loc[ids, COLS].join(b.loc[ids, COLS], lsuffix="_a", rsuffix="_b")
    col_wins = pd.DataFrame(
        {
            c: (both[f"{c}_a"] > both[f"{c}_b"]).where(
                both[f"{c}_a"].notna() & both[f"{c}_b"].notna()
            )
            for c in COLS
        }
    )
    majority = col_wins.sum(axis=1, min_count=1) >= 2
    return col_wins, majority[col_wins.notna().any(axis=1)]


def verdict(a, b, ids, label):
    ids = a.index.intersection(b.index).intersection(ids)
    if len(ids) < 5:
        return f"{label}: n={len(ids)} (too few)"
    col_wins, maj = case_wins(a, b, ids)
    m = maj.to_numpy(float)
    bs = [m[rng.integers(0, len(m), len(m))].mean() for _ in range(1000)]
    cols = " | ".join(f"{c} {col_wins[c].mean():.2f}" for c in COLS)
    return (
        f"{label}: n={len(ids)} | case-win {maj.mean():.2f} "
        f"[{np.percentile(bs, 10):.2f}, {np.percentile(bs, 90):.2f}] | per column: {cols}"
    )


ids_2020 = case_meta[case_meta.transfer_season >= 2020].index
ids_all = case_meta.index
ids_small = case_meta.index[
    case_meta.index.isin(
        cases_all_t.assign(
            case_id=lambda d: (
                d.from_club_id.astype("int64").astype(str)
                + "_"
                + d.transfer_season.astype("int64").astype(str)
                + "_"
                + d.tm_player_id
            )
        )[cases_all_t.elo_rank > 6].case_id
    )
]

print("=== model vs the three baselines, per role (2020+ = out of tuning) ===")
for role, g in case_meta.groupby("dep_role"):
    print(f"\n--- {role} ---")
    role_ids = g.index
    for opp in ["actual", "market", "naive"]:
        print(
            verdict(
                entrants["model"], entrants[opp], role_ids.intersection(ids_2020), f"vs {opp} 2020+"
            )
        )
        print(verdict(entrants["model"], entrants[opp], role_ids, f"vs {opp} all   "))

print("\n=== the role tournament (defensive roles): which ordering buys better? ===")
for role in ["CB", "FB", "CM"]:
    role_ids = case_meta[case_meta.dep_role == role].index
    print(f"\n--- {role} ---")
    print(
        verdict(entrants["model"], entrants["output_all"], role_ids, "fit-first vs output-ranking")
    )
    if role in {"CB", "FB"}:
        print(
            verdict(
                entrants["model"], entrants["defence"], role_ids, "fit-first vs defence-activity"
            )
        )

print("\n=== smaller-club split (seller Elo rank > 6), model vs baselines, roles pooled ===")
for opp in ["actual", "market", "naive"]:
    print(verdict(entrants["model"], entrants[opp], ids_small, f"vs {opp}"))

print("\n=== per season: model vs actual, case-win rate (roles pooled) ===")
common = entrants["model"].index.intersection(entrants["actual"].index)
_, maj_all = case_wins(entrants["model"], entrants["actual"], common)
by_season = maj_all.groupby(case_meta.loc[maj_all.index, "transfer_season"]).agg(["mean", "size"])
print(by_season.round(2).to_string())

=== model vs the three baselines, per role (2020+ = out of tuning) ===

--- CB ---
vs actual 2020+: n=84 | case-win 0.46 [0.40, 0.54] | per column: minutes_per_meur 0.60 | ga90_mean 0.54 | value_ratio 0.32
vs actual all   : n=108 | case-win 0.49 [0.43, 0.56] | per column: minutes_per_meur 0.63 | ga90_mean 0.53 | value_ratio 0.36
vs market 2020+: n=152 | case-win 0.73 [0.68, 0.78] | per column: minutes_per_meur 0.97 | ga90_mean 0.56 | value_ratio 0.49
vs market all   : n=237 | case-win 0.74 [0.71, 0.78] | per column: minutes_per_meur 0.98 | ga90_mean 0.54 | value_ratio 0.52
vs naive 2020+: n=152 | case-win 0.48 [0.43, 0.53] | per column: minutes_per_meur 0.39 | ga90_mean 0.51 | value_ratio 0.56
vs naive all   : n=237 | case-win 0.48 [0.43, 0.52] | per column: minutes_per_meur 0.41 | ga90_mean 0.45 | value_ratio 0.55

--- CM ---
vs actual 2020+: n=93 | case-win 0.51 [0.44, 0.57] | per column: minutes_per_meur 0.71 | ga90_mean 0.60 | value_ratio 0.38
vs actual all   : n=143 | case-win 0.4

fit-first vs defence-activity: n=227 | case-win 0.43 [0.39, 0.47] | per column: minutes_per_meur 0.35 | ga90_mean 0.49 | value_ratio 0.51

--- FB ---


fit-first vs output-ranking: n=185 | case-win 0.40 [0.35, 0.44] | per column: minutes_per_meur 0.49 | ga90_mean 0.27 | value_ratio 0.47
fit-first vs defence-activity: n=174 | case-win 0.48 [0.43, 0.53] | per column: minutes_per_meur 0.46 | ga90_mean 0.60 | value_ratio 0.47

--- CM ---
fit-first vs output-ranking: n=298 | case-win 0.40 [0.37, 0.43] | per column: minutes_per_meur 0.51 | ga90_mean 0.21 | value_ratio 0.58

=== smaller-club split (seller Elo rank > 6), model vs baselines, roles pooled ===
vs actual: n=305 | case-win 0.42 [0.39, 0.46] | per column: minutes_per_meur 0.52 | ga90_mean 0.58 | value_ratio 0.34
vs market: n=601 | case-win 0.72 [0.70, 0.75] | per column: minutes_per_meur 0.94 | ga90_mean 0.51 | value_ratio 0.51
vs naive: n=601 | case-win 0.53 [0.51, 0.56] | per column: minutes_per_meur 0.57 | ga90_mean 0.43 | value_ratio 0.56

=== per season: model vs actual, case-win rate (roles pooled) ===
                 mean  size
transfer_season            
2015             0

### Step 5 — what we got

**Vs the market ordering: decisive wins everywhere.** Case-win 0.55–0.79 per position (2020+),
0.72 [0.70, 0.75] pooled on smaller-club sellers. Buying the most expensive affordable player
is reliably beaten in every position.

**Vs the naive goals+assists rule: a modest edge, not a rout** — with a caveat in the naive
rule's favour: it also benefits from the similarity gate and the budget cap here, so it is far
stronger than a raw league table. CM 0.57, ST 0.57, W 0.54 for the system; CB 0.48, GK 0.46,
FB 0.36 against.

**Vs the clubs' actual signings: parity, with a clear pattern.** Pooled smaller-club case-win
0.42; at or above 0.50 in six of ten seasons (2021 an outlier at 0.27). The columns tell the
story: the system's shortlists deliver **more minutes per euro** (0.52–0.73) and **more
output** (0.54–0.68 in attacking positions), but **lose value growth** (0.26–0.44) — real
clubs systematically buy young appreciating assets. Matching professional scouting
departments while beating the market is the honest headline.

**The sorting head-to-head, first verdict.** Within the same profile-gated pools, the
output-led rule beats the similarity-led one at CB 0.43 [0.38, 0.46], FB 0.40 [0.35, 0.44],
CM 0.40 [0.37, 0.43] (numbers are the similarity side's case-wins), and a pure
defending-stats rule does no better. Two caveats stated plainly: the goals+assists column is
an output-flavoured referee — no measurable outcome rewards defending itself — and the one
market-judgment column, value growth, is where the similarity side *wins* at CB (0.54) and
CM (0.58). The question continues through Steps 7–9.


## Step 6 — Can defender quality be measured better? Percentages and possession-adjustment

**What?** Two stat families that might capture *how well* a defender defends rather than how
much defending he does: quality percentages (duel win rates, times dribbled past, errors,
possession lost) and possession-adjusted volumes (tackles scaled by how much defending the
team forced on him). Screened with the standard bars: a stat must repeat year to year
(r ≥ 0.3) and survive a club change.

**Why?** Raw defending counts are siege-biased — bad teams defend more. If rates fix that,
they belong in the player cards and the similarity gate; if not, that is worth knowing too.


In [15]:
# Step 6 — screen: stability and travel of quality percentages and padj volumes (CB, FB, CM)
from scout.models import quantities
from scout.panel import team_season

QUALITY = {
    "groundDuelsWonPercentage": "pct",
    "aerialDuelsWonPercentage": "pct",
    "accuratePassesPercentage": "pct",
    "dribbledPast": "per90",
    "errorLeadToShot": "per90",
    "possessionLost": "per90",
    "wasFouled": "per90",
}
VOLUMES = ["tackles", "interceptions", "clearances"]

q = ss[
    ["competition_id", "season", "sofascore_player_id", "minutesPlayed"] + list(QUALITY) + VOLUMES
].copy()
q["minutes"] = pd.to_numeric(q.minutesPlayed)
q["tm_player_id"] = q.sofascore_player_id.astype(int).astype(str).map(ss_ids)
q = q.dropna(subset=["tm_player_id"])
for col, kind in {**QUALITY, **{v: "per90" for v in VOLUMES}}.items():
    q[col] = pd.to_numeric(q[col], errors="coerce")
    if kind == "per90":
        q[col] = q[col] / q.minutes * 90
q = (
    q[q.minutes >= quantities.MIN_MINUTES]
    .sort_values(
        ["minutes", "competition_id", "sofascore_player_id"], ascending=[False, True, True]
    )
    .drop_duplicates(["tm_player_id", "season"])
)

# team defensive exposure: deep completions allowed per match, relative to the league-season mean
ts = team_season.build()
ts["competition_id"] = ts.league.map(LEAGUE_TO_COMP)
team_club = (
    us[["competition_id", "team", "team_id"]]
    .drop_duplicates()
    .merge(
        lineage[["competition_id", "team_name", "club_id"]].rename(columns={"team_name": "team"}),
        on=["competition_id", "team"],
    )
)
ts = ts.merge(team_club[["competition_id", "team_id", "club_id"]], on=["competition_id", "team_id"])
expo = ts[["competition_id", "season", "club_id", "deep_completions_against"]].copy()
expo["exposure"] = expo.deep_completions_against / expo.groupby(
    ["competition_id", "season"]
).deep_completions_against.transform("mean")

club_of = season_value[["tm_player_id", "season", "club_id"]].drop_duplicates(
    ["tm_player_id", "season"]
)
q = q.merge(club_of, on=["tm_player_id", "season"], how="left").merge(
    expo[["season", "club_id", "exposure"]], on=["season", "club_id"], how="left"
)
for v in VOLUMES:
    q[f"padj_{v}"] = q[v] / q.exposure

roles_map = pd.concat([dep_role, gk_role], ignore_index=True).rename(
    columns={"prev_season": "season", "dep_role": "role"}
)
q = q.merge(roles_map, on=["tm_player_id", "season"])
q = q[q.role.isin(["CB", "FB", "CM"])]

CANDS = list(QUALITY) + VOLUMES + [f"padj_{v}" for v in VOLUMES]
# data sanity: units and ranges per field (percentages must be 0-100, per-90s plausible)
print("field ranges (CB/FB/CM rows):")
print(
    q[CANDS]
    .describe(percentiles=[0.1, 0.5, 0.9])
    .T[["mean", "10%", "50%", "90%", "max"]]
    .round(2)
    .to_string()
)
print("\nmeans by role:")
print(q.groupby("role")[CANDS].mean().round(2).to_string())

print("coverage (share non-NaN, CB/FB/CM rows, n=%d):" % len(q))
print(q[CANDS].notna().mean().round(2).to_string())

nxt_q = q.assign(season=q.season - 1)[["tm_player_id", "season", "role", "club_id"] + CANDS]
paired = q.merge(nxt_q, on=["tm_player_id", "season", "role"], suffixes=("", "_next"))
paired["moved"] = paired.club_id != paired.club_id_next
for label, sub in [
    ("stability (all pairs)", paired),
    ("travel (movers only)", paired[paired.moved]),
]:
    print(f"\n=== {label} ===")
    out = {}
    for role, g in sub.groupby("role"):
        out[role] = {
            c: round(g[c].corr(g[f"{c}_next"]), 2) for c in CANDS if g[c].notna().sum() > 50
        }
    print(pd.DataFrame(out).round(2).to_string())
    print("n pairs:", sub.groupby("role").size().to_dict())

field ranges (CB/FB/CM rows):
                               mean       10%        50%        90%        max
groundDuelsWonPercentage  54.062016    44.724      53.93      63.54      88.24
aerialDuelsWonPercentage  52.309651     35.94      53.57      66.67      92.31
accuratePassesPercentage  82.252107    73.942      82.88      89.69      96.36
dribbledPast               0.893255  0.324824   0.794451   1.603007   3.782712
errorLeadToShot            0.030659       0.0        0.0    0.09311   0.401427
possessionLost             9.949322  2.032578  10.242131  16.051363  29.632075
wasFouled                  0.973387  0.340909   0.870968   1.756283   6.189402
tackles                    1.939299  1.083041   1.845943   2.913992   5.853659
interceptions               1.40692   0.67475   1.288253   2.309426   5.228216
clearances                 2.585865  0.791433   2.178423   5.005113  10.812721
padj_tackles               2.028194  1.035061   1.874329   3.225711   7.895731
padj_interceptions    

### Step 6 — what we got

**The data check passed** (units sane: percentages 0–100, per-90s plausible, role means make
football sense — CBs win the most aerials, CMs get dribbled past most) and the screen found
real defender-quality signals: **aerial and ground duel win rates, dribbled-past, possession
lost and fouls won all repeat (r 0.36–0.74) and travel across clubs (0.35–0.65)** — they join
the defender fingerprint and cards. Two clean negative findings alongside: **errors leading to
shots is luck** (r 0.12–0.13 — an "error-prone defender" is mostly a narrative), and
**possession-adjusting volumes makes them travel worse** (padj tackles CB 0.40 vs raw 0.47) —
the classic correction imports team-context noise, so raw per-90s stay. Sorting shortlists by
these quality stats was also graded and does not pick better signings than output — they
describe players; they don't rank them.


## Step 7 — The similarity × output blend

**What?** Equal rank-mixes of similarity and quality for the defensive positions — no
invented weights: similarity + probability of clearing the bar; the same plus expected
minutes; the same plus duel-quality. Tuned among themselves on pre-2020 cases, the winner
graded against the standing orderings.

**Why?** Step 5 tested pure similarity and pure output — never the blend of both, which is
the natural compromise candidate. An ordering defined up front and graded on outcomes is how
every entrant earns its seat.


In [16]:
# Step 7 — the blend: quality frame, candidates, pre-2020 tuning, then the tournament
QUALITY_SET = [
    "aerialDuelsWonPercentage",
    "groundDuelsWonPercentage",
    "dribbledPast",
    "possessionLost",
]
qual = q.drop_duplicates(["tm_player_id", "season"])[["tm_player_id", "season"] + QUALITY_SET]
cp = cand_pools[cand_pools.dep_role.isin(["CB", "FB", "CM"])].merge(
    qual.assign(transfer_season=qual.season + 1).drop(columns=["season"]),
    on=["tm_player_id", "transfer_season"],
    how="left",
)
for col, sign in [
    ("aerialDuelsWonPercentage", 1),
    ("groundDuelsWonPercentage", 1),
    ("dribbledPast", -1),
    ("possessionLost", -1),
]:
    cp[f"z_{col}"] = sign * cp.groupby(["dep_role", "transfer_season"])[col].transform(
        lambda x: (x - x.mean()) / (x.std() if x.std() > 0 else 1.0)
    )
cp["qual_z"] = cp[[f"z_{c}" for c in QUALITY_SET]].mean(axis=1)


def blend_scores(pool_subset, keys, n=5):
    rows = []
    for cid, g in pool_subset.groupby("case_id"):
        ranks = sum((g[k] * sign).rank() for k, sign in keys) / len(keys)
        top = g.assign(_k=ranks).dropna(subset=["_k"]).nsmallest(n, "_k")
        if len(top) < n:
            continue
        rows.append(
            {
                "case_id": cid,
                "minutes_per_meur": top.out_minutes.fillna(0).sum() / (top.value.sum() / 1e6),
                "ga90_mean": top.out_ga90.mean(),
                "value_ratio": top.out_value_next.sum() / top.value.sum(),
            }
        )
    return pd.DataFrame(rows).set_index("case_id")


BLENDS = {
    "sim+out": [("dist", 1), ("p_bar", -1)],
    "sim+out+min": [("dist", 1), ("p_bar", -1), ("expected_minutes", -1)],
    "sim+out+qual": [("dist", 1), ("p_bar", -1), ("qual_z", -1)],
}
pre_cp = cp[cp.transfer_season <= 2019]
tuning = {}
for name, keys in BLENDS.items():
    sc = blend_scores(pre_cp, keys)
    tuning[name] = sc[["minutes_per_meur", "ga90_mean", "value_ratio"]].mean()
tuning = pd.DataFrame(tuning).T
print("=== blend tuning, pre-2020 cases (CB/FB/CM, top-5) ===")
print(tuning.round(3).to_string())
winner_blend = tuning.rank(ascending=False).mean(axis=1).idxmin()
print("winner by mean rank:", winner_blend)

entrants["blend"] = blend_scores(cp, BLENDS[winner_blend])
print("\n=== the winning blend vs the field (POST-HOC candidate — promising, not final) ===")
for role in ["CB", "FB", "CM"]:
    ids = case_meta[case_meta.dep_role == role].index
    print(f"\n--- {role} ---")
    print(verdict(entrants["blend"], entrants["output_all"], ids, "blend vs output-ranking (all)"))
    print(
        verdict(
            entrants["blend"],
            entrants["output_all"],
            ids.intersection(ids_2020),
            "blend vs output-ranking 2020+",
        )
    )
    print(verdict(entrants["blend"], entrants["model"], ids, "blend vs fit-first (all)     "))
    print(verdict(entrants["blend"], entrants["actual"], ids, "blend vs actual signing (all)"))

=== blend tuning, pre-2020 cases (CB/FB/CM, top-5) ===
              minutes_per_meur  ga90_mean  value_ratio
sim+out                574.167      0.137        1.025
sim+out+min            563.077      0.137        1.098
sim+out+qual           536.962      0.129        1.029
winner by mean rank: sim+out



=== the winning blend vs the field (POST-HOC candidate — promising, not final) ===

--- CB ---
blend vs output-ranking (all): n=237 | case-win 0.45 [0.41, 0.49] | per column: minutes_per_meur 0.53 | ga90_mean 0.43 | value_ratio 0.45
blend vs output-ranking 2020+: n=152 | case-win 0.47 [0.42, 0.53] | per column: minutes_per_meur 0.56 | ga90_mean 0.47 | value_ratio 0.46
blend vs fit-first (all)     : n=237 | case-win 0.51 [0.47, 0.55] | per column: minutes_per_meur 0.59 | ga90_mean 0.46 | value_ratio 0.47
blend vs actual signing (all): n=108 | case-win 0.52 [0.46, 0.58] | per column: minutes_per_meur 0.69 | ga90_mean 0.58 | value_ratio 0.34

--- FB ---
blend vs output-ranking (all): n=185 | case-win 0.42 [0.37, 0.46] | per column: minutes_per_meur 0.46 | ga90_mean 0.33 | value_ratio 0.50
blend vs output-ranking 2020+: n=100 | case-win 0.40 [0.34, 0.46] | per column: minutes_per_meur 0.42 | ga90_mean 0.32 | value_ratio 0.51
blend vs fit-first (all)     : n=185 | case-win 0.58 [0.54, 0.62

### Step 7 — what we got

**The blend beats similarity-first orderings, but not output.** Among the three blends,
similarity+output won the pre-2020 tuning (the duel-quality term only dragged). Against the
field: it clearly improves on similarity+minutes (FB 0.58, CB 0.51, CM 0.50) and reaches
parity with the clubs' actual signings at CB/CM (0.52/0.50), but against output-within-the-gate
it stays under 0.5 in every defensive role (CB 0.45, FB 0.42, CM 0.41). **Kept as the
similarity-respecting alternative:** shortlist tops that visibly resemble the departed player,
at a cost of a few points of case-win rate.


## Step 8 — The Moneyball formula: expected production, and production per euro

**What?** Two orderings built from the project's own stated objective rather than a new
invention: **expected season production** (quality per 90 × expected minutes — what a player
will actually deliver on the pitch, not just his rate) and **surplus production per euro**
(production above what a freely-available player gives, divided by price).

**Why?** Every mix tested so far *averaged* signals, which dilutes the strongest one. These
two multiply instead: quality × availability is a physical quantity, not a compromise — and
the per-euro form is the design spec's objective sentence, written before any data was
touched, finally used as the sort. Identified late, so the post-hoc caveat applies; future
seasons are the clean confirmation.


In [17]:
# Step 8 — the two production keys, graded (defensive roles)
srp = (
    contrib[["player_id", "season", "role", "surplus"]]
    .drop_duplicates(["player_id", "season", "role"])
    .set_index(["player_id", "season", "role"])
    .surplus
)
cp7 = cp.copy()
cp7["surplus_pt"] = srp.reindex(
    pd.MultiIndex.from_arrays([cp7.player_id, cp7.transfer_season - 1, cp7.dep_role])
).to_numpy()
cp7["production"] = cp7.point * cp7.expected_minutes / 90
cp7["prod_per_eur"] = (cp7.surplus_pt * cp7.expected_minutes / 90) / (cp7.value / 1e6)
print(
    "key coverage:",
    {c: f"{cp7[c].notna().mean():.1%}" for c in ["production", "prod_per_eur"]},
)

entrants["production"] = blend_scores(cp7, [("production", -1)])
entrants["prod_per_eur"] = blend_scores(cp7, [("prod_per_eur", -1)])


print("\n=== the declared three vs output-within-gate and vs actual (post-hoc, spec-native) ===")
for name in ["production", "prod_per_eur"]:
    print(f"\n--- {name} ---")
    for role in ["CB", "FB", "CM"]:
        ids = case_meta[case_meta.dep_role == role].index
        print(verdict(entrants[name], entrants["output_all"], ids, f"{role} vs output"))
        print(verdict(entrants[name], entrants["actual"], ids, f"{role} vs actual"))

print("\n=== value-growth column vs output-ranking ===")
for name in ["production", "prod_per_eur"]:
    row = []
    for role in ["CB", "FB", "CM"]:
        ids = case_meta[case_meta.dep_role == role].index
        common = entrants[name].index.intersection(entrants["output_all"].index).intersection(ids)
        cols, _ = case_wins(entrants[name], entrants["output_all"], common)
        row.append(f"{role} {cols.value_ratio.mean():.2f}")
    print(f"{name:14s} " + " | ".join(row))

key coverage: {'production': '100.0%', 'prod_per_eur': '100.0%'}



=== the declared three vs output-within-gate and vs actual (post-hoc, spec-native) ===

--- production ---
CB vs output: n=237 | case-win 0.46 [0.41, 0.50] | per column: minutes_per_meur 0.37 | ga90_mean 0.54 | value_ratio 0.49
CB vs actual: n=108 | case-win 0.48 [0.43, 0.55] | per column: minutes_per_meur 0.64 | ga90_mean 0.56 | value_ratio 0.36
FB vs output: n=185 | case-win 0.46 [0.41, 0.50] | per column: minutes_per_meur 0.47 | ga90_mean 0.38 | value_ratio 0.52
FB vs actual: n=80 | case-win 0.50 [0.42, 0.56] | per column: minutes_per_meur 0.64 | ga90_mean 0.55 | value_ratio 0.39
CM vs output: n=298 | case-win 0.57 [0.53, 0.61] | per column: minutes_per_meur 0.54 | ga90_mean 0.45 | value_ratio 0.58
CM vs actual: n=143 | case-win 0.50 [0.44, 0.55] | per column: minutes_per_meur 0.69 | ga90_mean 0.64 | value_ratio 0.38

--- prod_per_eur ---
CB vs output: n=237 | case-win 0.59 [0.55, 0.63] | per column: minutes_per_meur 0.97 | ga90_mean 0.46 | value_ratio 0.38
CB vs actual: n=108 | ca

### Step 8 — what we got: the Moneyball formula wins

**Surplus production per euro beats output-within-the-gate in all three defensive roles:**
CB 0.59 [0.55, 0.63], FB 0.52 [0.48, 0.57], CM 0.58 [0.54, 0.62] — CB and CM with intervals
clear of even. It is also the first ordering to convincingly beat the clubs' actual signings:
CB 0.56, FB 0.53, **CM 0.62 [0.56, 0.66]**. It wins by buying enormous delivered contribution
per euro (minutes column 0.86–0.97) while holding the output column against real clubs; like
every ordering except coach-trust-style ones, it concedes value growth. Plain production
(not per euro) is mixed (CM 0.57, CB/FB 0.46). Taken seriously but carefully: the formula was
graded after the main tournament — yet it is the design spec's pre-data objective, not a
shape fitted to this table; 2025-26 onward is its untouched confirmation set.


## Step 9 — The formula across all six positions

**What?** The Step 8 winner extended to wingers, strikers and goalkeepers (keeper surplus
from the goals-prevented replacement level), graded against each position's standing ordering
and against the clubs' actual signings — the complete six-position table, and the final
design of the ranked number.


In [18]:
# Step 9 — the formula across all six roles
srp_all = (
    pd.concat(
        [
            contrib[["player_id", "season", "role", "surplus"]],
            gk_universe[["player_id", "season", "surplus"]].assign(role="GK"),
        ],
        ignore_index=True,
    )
    .drop_duplicates(["player_id", "season", "role"])
    .set_index(["player_id", "season", "role"])
    .surplus
)

cpA = cand_pools.copy()
cpA["surplus_pt"] = srp_all.reindex(
    pd.MultiIndex.from_arrays([cpA.player_id, cpA.transfer_season - 1, cpA.dep_role])
).to_numpy()
cpA["prod_per_eur"] = (cpA.surplus_pt * cpA.expected_minutes / 90) / (cpA.value / 1e6)
print(f"key coverage all roles: {cpA.prod_per_eur.notna().mean():.1%} of {len(cpA)} rows")

entrants["prod_per_eur_all"] = blend_scores(cpA, [("prod_per_eur", -1)])

print("\n=== surplus production per euro vs the standing ordering and vs actual, all six roles ===")
for role in ["GK", "CB", "FB", "CM", "W", "ST"]:
    ids = case_meta[case_meta.dep_role == role].index
    print(f"\n--- {role} ---")
    print(verdict(entrants["prod_per_eur_all"], entrants["output_all"], ids, "vs output-ranking"))
    print(verdict(entrants["prod_per_eur_all"], entrants["actual"], ids, "vs actual signing"))

key coverage all roles: 100.0% of 60411 rows



=== surplus production per euro vs the standing ordering and vs actual, all six roles ===

--- GK ---
vs output-ranking: n=78 | case-win 0.26 [0.19, 0.32] | per column: minutes_per_meur 0.90 | ga90_mean 0.00 | value_ratio 0.26
vs actual signing: n=32 | case-win 0.25 [0.16, 0.34] | per column: minutes_per_meur 0.84 | ga90_mean 0.00 | value_ratio 0.29

--- CB ---
vs output-ranking: n=237 | case-win 0.59 [0.56, 0.64] | per column: minutes_per_meur 0.97 | ga90_mean 0.46 | value_ratio 0.38
vs actual signing: n=108 | case-win 0.56 [0.50, 0.63] | per column: minutes_per_meur 0.89 | ga90_mean 0.51 | value_ratio 0.34

--- FB ---
vs output-ranking: n=185 | case-win 0.52 [0.48, 0.57] | per column: minutes_per_meur 0.92 | ga90_mean 0.27 | value_ratio 0.43
vs actual signing: n=80 | case-win 0.53 [0.46, 0.60] | per column: minutes_per_meur 0.86 | ga90_mean 0.50 | value_ratio 0.31

--- CM ---
vs output-ranking: n=298 | case-win 0.58 [0.54, 0.62] | per column: minutes_per_meur 0.92 | ga90_mean 0.35 |

### Step 9 — what we got: one formula for the outfield, keepers exempt

**Surplus production per euro wins or edges every outfield role** — vs the standing ordering:
W 0.65 [0.61, 0.69], CB 0.59, CM 0.58, ST 0.55, FB 0.52; vs the clubs' actual signings:
W 0.66 [0.61, 0.71], CM 0.62, CB 0.56, FB 0.53, ST 0.49. Wingers, unexpectedly, are its
strongest role; the persistent weakness stays value growth — the market's appreciation game
remains the one column real clubs win. **Goalkeepers are the exception** (0.26/0.25: the GK
surplus rests on the noisier prevented proxy, and keepers produce no G+A so their verdict is
effectively two-column) — GK keeps P(≥ bar) on the prevented proxy.

**The final design of the ranked number, pending future-season confirmation:** filter to the
departing player's profile (the football knowledge), then rank outfield candidates by
(point − replacement) × expected minutes ÷ price; GK on the prevented-proxy probability;
output-within-gate and the similarity blend kept as the graded alternatives. A number of
further ordering variants were explored during development and discarded as clearly worse;
the committed artifact retains the graded set.


## Step 10 — The package reproduces the backtest

**What?** `python -m scout train backtest` rebuilds everything above from source and writes
`models/phase5_backtest.json` — the committed file every quoted number traces to.

**One correction the reproduction caught:** the exploratory pools above had kept the 245
relegated-seller cases that Step 1 decided to exclude (1,304 cases vs the decision-faithful
1,059). The package implements the decision; the smaller-club split — which never contained
those sellers — matches exactly, and every other verdict moves by at most ±0.03 with no
conclusion changing. The artifact is the source of truth for the writeup.


In [19]:
# Step 10 check — the committed artifact's headline numbers
art = json.load(open(config.MODELS / "phase5_backtest.json"))
print("population:", art["population"])
print("decisions:", art["decisions"])
for role in ["CB", "CM", "FB", "GK", "ST", "W"]:
    v = art["verdicts"][role]["actual"]["out_of_tuning_2020_plus"]
    m = art["verdicts"][role]["market"]["all_seasons"]
    print(f"{role}: vs actual 2020+ {v} | vs market all {m}")
print("tournament:", art["tournament_defensive_roles"])
print("smaller club:", art["smaller_club_split_elo_rank_gt_6"])
print("formula, all roles:", art["formula_all_roles"])
print("per season vs actual:", art["per_season_vs_actual"])

population: {'cases_scored': 1059, 'pool_rows': 49939, 'cases_with_actual_signing': 559, 'seasons': [2015, 2024], 'note': 'summer paid departures, >=600-min role season, top-flight Big-5 sellers; Big-5 candidate pools'}
decisions: {'gate': 'top-50 most similar within role', 'budget': "candidate value <= the departing player's sale fee", 'orderings': {'CB/FB/CM': 'f3', 'W/ST/GK': 'o2'}, 'probability': 'normal on the Phase 2 interval', 'shortlist': 5}
CB: vs actual 2020+ {'n': 83, 'case_win': 0.4699, 'lo': 0.4084, 'hi': 0.5422, 'columns': {'minutes_per_meur': 0.5904, 'ga90_mean': 0.5385, 'value_ratio': 0.3243}} | vs market all {'n': 190, 'case_win': 0.7474, 'lo': 0.7053, 'hi': 0.7895, 'columns': {'minutes_per_meur': 0.9789, 'ga90_mean': 0.5474, 'value_ratio': 0.5158}}
CM: vs actual 2020+ {'n': 91, 'case_win': 0.5165, 'lo': 0.4505, 'hi': 0.5824, 'columns': {'minutes_per_meur': 0.7033, 'ga90_mean': 0.5977, 'value_ratio': 0.3837}} | vs market all {'n': 250, 'case_win': 0.72, 'lo': 0.684, 'h